# Lab 1: Customer Support Agent using Strands Agents

## Overview

[Strands Agents](https://strandsagents.com/latest/) is a simple yet powerful SDK that takes a model-driven approach to building and running AI agents.

In this lab, you'll create a **Customer Support Agent** using the Strands Agents framework. We will use this Agent to explore the capabilities provided by Amazon Bedrock AgentCore in the following Labs.

The **Customer Support Agent** has the following tools available:
- **get_return_policy()** - Get return policy for specific products
- **get_product_info()** - Get product information
- **web_search()** - Search the web for troubleshooting help

As a safety measure, to control our Agent's behaviour, let's add [**Bedrock Guardrails**](https://aws.amazon.com/bedrock/guardrails/) to the Agent to block any unwanted topics.

## Architecture
<div style="text-align:left">
    <img src="images/architecture_lab1_strands.png" width="75%"/>
</div>


## Prerequisites

* Python 3.12+
* AWS credentials configured  
* Anthropic Claude 3.7 enabled on [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* Strands Agents and supporting libraries


## Step 1: Install Dependencies and Import Libraries

In [ ]:
# Install required packages
%pip install -r requirements.txt -q

In [ ]:
# Import libraries
import boto3
from boto3.session import Session

from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

from strands.tools import tool
from scripts.utils import put_ssm_parameter

In [ ]:
# Get boto session
boto_session = Session()
region = boto_session.region_name

## Step 2: Implementing custom tools

Next, we will implement the 3 tools which will be provided to the Customer Support Agent.

Defining tools in Strands Agent is extremely simple, just add a `@tool` decorator to your function, and provide a description of the tool in the function's docstring. Strands Agents will use the function documentation, typing and arguments to provide context on this tool to your agent. 


### Tool 1: Get Return Policy

**Purpose:** This tool helps customers understand return policies for different product categories. It provides detailed information about return windows, conditions, processes, and refund timelines so customers know exactly what to expect when returning items.

In [ ]:
@tool
def get_return_policy(product_category: str) -> str:
    """
    Get return policy information for a specific product category.

    Args:
        product_category: Electronics category (e.g., 'smartphones', 'laptops', 'accessories')

    Returns:
        Formatted return policy details including timeframes and conditions
    """
    # Mock return policy database - in real implementation, this would query policy database
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included"
        },
         "laptops": {
            "window": "30 days", 
            "condition": "Original packaging, all accessories, no software modifications",
            "process": "Technical support verification required before return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping with original packaging",
            "warranty": "1-year manufacturer warranty, extended options available"
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "process": "Online return portal",
            "refund_time": "3-5 business days after receipt",
            "shipping": "Customer pays return shipping under $50",
            "warranty": "90-day manufacturer warranty"
        }
    }

    # Default policy for unlisted categories
    default_policy = {
        "window": "30 days",
        "condition": "Original condition with all included components",
        "process": "Contact technical support",
        "refund_time": "5-7 business days after inspection", 
        "shipping": "Return shipping policies vary",
        "warranty": "Standard manufacturer warranty applies"
    }

    policy = return_policies.get(product_category.lower(), default_policy)
    return f"Return Policy - {product_category.title()}:\n\n" \
           f"• Return window: {policy['window']} from delivery\n" \
           f"• Condition: {policy['condition']}\n" \
           f"• Process: {policy['process']}\n" \
           f"• Refund timeline: {policy['refund_time']}\n" \
           f"• Shipping: {policy['shipping']}\n" \
           f"• Warranty: {policy['warranty']}"
print("✅ Return policy tool ready")

### Tool 2: Get Product Information

**Purpose:** This tool provides customers with comprehensive product details including warranties, available models, key features, shipping policies, and return information. It helps customers make informed purchasing decisions and understand what they're buying.

In [ ]:
@tool
def get_product_info(product_type: str) -> str:
    """
    Get detailed technical specifications and information for electronics products.

    Args:
        product_type: Electronics product type (e.g., 'laptops', 'smartphones', 'headphones', 'monitors')
    Returns:
        Formatted product information including warranty, features, and policies
    """
    # Mock product catalog - in real implementation, this would query a product database
    products = {
        "laptops": {
            "warranty": "1-year manufacturer warranty + optional extended coverage",
            "specs": "Intel/AMD processors, 8-32GB RAM, SSD storage, various display sizes",
            "features": "Backlit keyboards, USB-C/Thunderbolt, Wi-Fi 6, Bluetooth 5.0",
            "compatibility": "Windows 11, macOS, Linux support varies by model",
            "support": "Technical support and driver updates included"
        },
        "smartphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "5G/4G connectivity, 128GB-1TB storage, multiple camera systems",
            "features": "Wireless charging, water resistance, biometric security",
            "compatibility": "iOS/Android, carrier unlocked options available",
            "support": "Software updates and technical support included"
        },
        "headphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "Wired/wireless options, noise cancellation, 20Hz-20kHz frequency",
            "features": "Active noise cancellation, touch controls, voice assistant",
            "compatibility": "Bluetooth 5.0+, 3.5mm jack, USB-C charging",
            "support": "Firmware updates via companion app"
        },
        "monitors": {
            "warranty": "3-year manufacturer warranty",
            "specs": "4K/1440p/1080p resolutions, IPS/OLED panels, various sizes",
            "features": "HDR support, high refresh rates, adjustable stands",
            "compatibility": "HDMI, DisplayPort, USB-C inputs",
            "support": "Color calibration and technical support"
        }
    }
    product = products.get(product_type.lower())
    if not product:
        return f"Technical specifications for {product_type} not available. Please contact our technical support team for detailed product information and compatibility requirements."

    return f"Technical Information - {product_type.title()}:\n\n" \
           f"• Warranty: {product['warranty']}\n" \
           f"• Specifications: {product['specs']}\n" \
           f"• Key Features: {product['features']}\n" \
           f"• Compatibility: {product['compatibility']}\n" \
           f"• Support: {product['support']}"

print("✅ get_product_info tool ready")

### Tool 3: Web-search

**Purpose:** This tool allows customers to get troubleshooting support or suggestions on product recommendations etc.

In [ ]:
@tool
def web_search(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.
    
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"

print("✅ Web search tool ready")

## Step 3: Create Bedrock Guardrails 

For illustrating the use of guardarails with our customer support assistant, we'll create a guardrail to simulate topics we don't want our chatbot to respond to, notably on finance - any question or instruction related to financial information, transactions, or related. [Read more about guardrails here](https://strandsagents.com/latest/documentation/docs/user-guide/safety-security/guardrails/)

In [ ]:
from lab_helpers.lab1_guardrails import create_or_get_guardrail_resource

guardrail_id, guardrail_version = create_or_get_guardrail_resource()

In [ ]:
print("The guardrailId is", guardrail_id)
print("The guardrail version is", guardrail_version)

## Step 4: Create and Configure the Customer Support 

Next, we will create the Customer Support Agent providing a model, the list of tools implemented in the previous step, a system prompt, and the created Bedrock Guardrail resource.


In [ ]:
from strands import Agent
from strands.models import BedrockModel

SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information and product specifications.
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For warranty and return policy questions
2. get_product_info() - To get information about a specific product
3. web_search() - To access current technical documentation, or for updated information. 
Always use the appropriate tool to get accurate, up-to-date information rather than making assumptions about electronic products or specifications."""

# Initialize the Bedrock model (Anthropic Claude 3.7 Sonnet)
model = BedrockModel(
    model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    temperature=0.3,
    region_name=region,
    guardrail_id=guardrail_id,
    # guardrail_version=guardrail_version,
    guardrail_trace="enabled",  # Enable trace info for debugging
)

# Create the customer support agent with all 5 tools
agent = Agent(
    model=model,
    tools=[
        get_product_info,  # Tool 1: Simple product information lookup
        get_return_policy,  # Tool 2: Simple return policy lookup
        web_search # Tool 3: Access the web for updated information
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("Customer Support Agent created successfully!")

## Step 5: Test the Customer Support Agent

Let's test our agent with sample queries to ensure all tools work correctly.

### Test Return check

In [ ]:
response = agent("What's the return policy for my thinkpad X1 Carbon?")

### Test Troubleshooting

In [ ]:
response = agent("I bought an iphone 14 last month. I don't like it because it heats up. How do I solve it?")

### Test Bedrock Guardrail

In [ ]:
response = agent("Where can I invest now ?")

## 🎉 Congratulations!

You've successfully created your first Strands agent secured with Bedrock Guardrails.
